# ODE - Hybrid Setting (Periodic)

In this notebook, we solve the periodic ODE
$$
\begin{aligned}
\partial_t u + \mu u &= f \quad \text{on } I=(0,T), \\
u(0) &= u(T).
\end{aligned}
$$
We use the variational formulation: find $u \in X_{\mathrm{per}}$ such that
$$
b(u,v)
= \langle \partial_t u,(\mathcal H_{\mathrm{per}}+I)v\rangle_I
+ \mu\langle u,(\mathcal H_{\mathrm{per}}+I)v\rangle_I
= \langle f,(\mathcal H_{\mathrm{per}}+I)v\rangle_I.
$$
Here $\mathcal H_{\mathrm{per}}$ is the periodic modified Hilbert transform with Fourier multiplier
$i\,\operatorname{sgn}(k)$ for $k\ne0$ and zero multiplier for $k=0$. With this sign convention,
$\mathcal H_{\mathrm{per}}\sin(2\pi t/T)=\cos(2\pi t/T)$.

The periodic trial and test space is $X_{\mathrm{per}}=H^{1/2}_{\mathrm{per}}(I)$, equipped with
$$
\lVert u\rVert_{X_{\mathrm{per}}}^2
= |u|_{H^{1/2}_{\mathrm{per}}(I)}^2
+ \mu\lVert u\rVert_{L^2(I)}^2.
$$
For $\mu>0$, the bilinear form is elliptic in this norm. We discretize with continuous,
piecewise polynomials of degree $p$ on a uniform partition and identify the degrees of freedom
at $t=0$ and $t=T$.

We need `hmod` and the following standard libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

We choose a smooth periodic manufactured solution with a nonzero mean:
$$
\begin{aligned}
u(t) &= 1 + \sin(2\pi t/T), \\
f(t) &= \partial_tu(t)+\mu u(t)
= \frac{2\pi}{T}\cos(2\pi t/T)+\mu\bigl(1+\sin(2\pi t/T)\bigr).
\end{aligned}
$$
The nonzero mean also exercises the zero Fourier mode, which is annihilated by
$\mathcal H_{\mathrm{per}}$ but controlled by the mass term.

In [ ]:
T = 5.0
mu = 3.0
periodic = True
omega = 2.0 * np.pi / T

u = lambda t: 1.0 + np.sin(omega * t)
dt_u = lambda t: omega * np.cos(omega * t)
f = lambda t: dt_u(t) + mu * u(t)

Next, we define the number of temporal intervals $n_t$, the polynomial degree $p$, and a hierarchy
of temporal grids for the preconditioner. A periodic continuous Lagrange space has $n_t p$ degrees
of freedom because the endpoints are identified.

In [ ]:
nt_coarsest = 2
refinements = 4
nt = nt_coarsest * 2**refinements
polynomial_degree = 3
n_periodic_dofs = nt * polynomial_degree

print(
    f"Number of temporal intervals: {nt}, "
    f"polynomial degree: {polynomial_degree}, "
    f"periodic Lagrange DOFs: {n_periodic_dofs}"
)

For the load vector, we first compute the $L^2$-projection of $f$ onto the discontinuous,
piecewise polynomial space in Legendre basis. This local projection is unchanged by the periodic
identification of the continuous Lagrange space.

In [ ]:
from hmod.standard_matrices import project_rhs_onto_legendre_basis

f_h = project_rhs_onto_legendre_basis(
    f,
    nt=nt,
    polynomial_degree_test=polynomial_degree,
    T=T,
    periodic=periodic,
)

We assemble the standard finite element matrices
$$
A_t \leftrightarrow \langle \partial_tu,v\rangle_I,
\qquad
M_t \leftrightarrow \langle u,v\rangle_I.
$$
Passing `periodic=True` identifies the endpoint degrees of freedom during the Lagrange-to-Legendre
transformation. In particular, both matrices have shape $(n_t p)\times(n_t p)$.

In [ ]:
from hmod.standard_matrices import get_lagrange_lagrange_matrix_for_derivatives

A_t = get_lagrange_lagrange_matrix_for_derivatives(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    T=T,
    derivatives_trial=1,
    derivatives_test=0,
    periodic=periodic,
)
M_t = get_lagrange_lagrange_matrix_for_derivatives(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    T=T,
    derivatives_trial=0,
    derivatives_test=0,
    periodic=periodic,
)

The terms containing the periodic modified Hilbert transform are
$$
A_{\mathcal H_{\mathrm{per}}}
\leftrightarrow \langle \partial_tu,\mathcal H_{\mathrm{per}}v\rangle_I,
\qquad
M_{\mathcal H_{\mathrm{per}}}
\leftrightarrow \langle u,\mathcal H_{\mathrm{per}}v\rangle_I.
$$
These matrices are dense in physical coordinates, so `hmod` exposes them as matrix-free SciPy
`LinearOperator`s. For the periodic transform, their application uses ordinary FFTs and
frequency-wise kernel multiplications.

In [ ]:
from hmod.hilbert_matrices import get_hilbert_matrix_for_derivatives_lagrange_lagrange

Ah_t = get_hilbert_matrix_for_derivatives_lagrange_lagrange(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    derivatives_trial=1,
    derivatives_test=0,
    T=T,
    periodic=periodic,
)
Mh_t = get_hilbert_matrix_for_derivatives_lagrange_lagrange(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    derivatives_trial=0,
    derivatives_test=0,
    T=T,
    periodic=periodic,
)

We convert the sparse standard matrices to linear operators and form the complete left-hand side.
There is no boundary degree of freedom to eliminate: periodicity is already built into the basis.

In [ ]:
M_t_op = spla.aslinearoperator(M_t)
A_t_op = spla.aslinearoperator(A_t)

B = Ah_t + A_t_op + mu * (Mh_t + M_t_op)
assert B.shape == (n_periodic_dofs, n_periodic_dofs)

The projected right-hand side is in the discontinuous Legendre basis, while the test space is the
continuous periodic Lagrange space. We therefore combine Legendre-Lagrange mass operators with and
without $\mathcal H_{\mathrm{per}}$.

In [ ]:
from hmod.standard_matrices import get_legendre_lagrange_matrix_for_derivatives
from hmod.hilbert_matrices import get_hilbert_matrix_for_derivatives_legendre_lagrange

M_legendre_lagrange = get_legendre_lagrange_matrix_for_derivatives(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    T=T,
    derivatives_trial=0,
    derivatives_test=0,
    periodic=periodic,
)
M_legendre_lagrange_op = spla.aslinearoperator(M_legendre_lagrange)
Mh_legendre_lagrange = get_hilbert_matrix_for_derivatives_legendre_lagrange(
    polynomial_degree_trial=polynomial_degree,
    polynomial_degree_test=polynomial_degree,
    nt=nt,
    derivatives_trial=0,
    derivatives_test=0,
    T=T,
    periodic=periodic,
)

M_rhs = M_legendre_lagrange_op + Mh_legendre_lagrange
rhs = M_rhs @ f_h
assert rhs.shape == (n_periodic_dofs,)

The system is elliptic in the periodic hybrid norm but non-symmetric, so we use GMRES. The periodic
BPX preconditioner uses periodic mass matrices and prolongation operators on every level; unlike the
initial-value case, it does not remove the first degree of freedom.

In [ ]:
from hmod.preconditioning import BPXPreconditioner, GMRESCounter

BPX = BPXPreconditioner(
    mu=mu,
    n_refinements=refinements,
    sobolev_exponent=0.5,
    polynomial_degree=polynomial_degree,
    nt_coarse=nt_coarsest,
    T=T,
    periodic=periodic,
)

In [ ]:
counter = GMRESCounter(print_residual=True)
u_h, info = spla.gmres(
    B, rhs, M=BPX, callback=counter, callback_type="pr_norm", rtol=1e-10
)

relative_residual = np.linalg.norm(B @ u_h - rhs) / np.linalg.norm(rhs)
print(f"GMRES finished with info code {info} after {counter.niter} iterations.")
print(f"Relative residual: {relative_residual:.2e}")
assert info == 0
assert relative_residual < 1e-9

To evaluate the numerical solution, we transform the periodic Lagrange vector to the discontinuous
Legendre representation. The periodic transformation has shape
$n_t(p+1)\times n_t p$ and maps the endpoint of the last element back to the first Lagrange degree
of freedom.

In [ ]:
from hmod.polynomial_bases import LegendreBasisEvaluator, get_lagrange_to_legendre_matrix

Trans = get_lagrange_to_legendre_matrix(
    polynomial_degree=polynomial_degree,
    nt=nt,
    periodic=periodic,
)
assert Trans.shape == (nt * (polynomial_degree + 1), n_periodic_dofs)

u_h_legendre = Trans @ u_h
evaluator = LegendreBasisEvaluator(
    dofs=u_h_legendre,
    polynomial_degree=polynomial_degree,
    nt=nt,
    T=T,
)

Finally, we compute the $L^2$ error, the periodic $H^{1/2}$ seminorm error, the corresponding hybrid
norm error, and the numerical periodicity defect.

In [ ]:
from hmod.norms import compute_l2_norm, compute_h12_seminorm

error_integrand = lambda t: evaluator.evaluate(t) - u(t)

L2_error = compute_l2_norm(error_integrand, nt=nt, T=T, periodic=periodic)
H1_2_error = compute_h12_seminorm(
    error_integrand,
    T=T,
    quad_tol=1e-8,
    periodic=periodic,
)
X_norm = np.sqrt(mu * L2_error**2 + H1_2_error**2)
periodicity_defect = abs(evaluator.evaluate(0.0) - evaluator.evaluate(T))

print(
    f"L2 error: {L2_error:.2e}, "
    f"periodic H1/2 seminorm error: {H1_2_error:.2e}, "
    f"X-norm error: {X_norm:.2e}"
)
print(f"Periodicity defect |u_h(0)-u_h(T)|: {periodicity_defect:.2e}")
assert periodicity_defect < 1e-12

We plot the numerical and analytical solutions over one period.

In [ ]:
t_plot = np.linspace(0.0, T, 1000)
u_plot = np.asarray(evaluator.evaluate(t_plot))
u_analytical_plot = u(t_plot)

plt.plot(t_plot, u_plot, label="Numerical solution")
plt.plot(t_plot, u_analytical_plot, label="Analytical solution", linestyle="dashed")
plt.legend()
plt.xlabel("t")
plt.ylabel("u(t)")
plt.title("Periodic numerical solution vs analytical solution")
plt.show()